# Expert Agent 5: StrandFracture Expert
## 반단선 (Semi-disconnection) 판별 전문가

이 노트북은 Langgraph와 Vertex AI Gemini 2.5 Flash를 사용하여 전기화재 감식 중 반단선 현상을 판별하는 전문가 에이전트를 구현합니다.

### 분석 프로세스
1. **소선 끝단의 형상 분석**: 네킹(Necking) 및 뾰족한 끝단 탐지
2. **용융망울의 크기와 분포**: 미세 망울(Micro-beads) 확인
3. **기계적 피로 흔적**: 반복적인 굽힘/비틀림 손상 확인

In [8]:
import base64
import json
import os
from typing import TypedDict, Annotated, Literal, List, Dict, Any, Optional
from pathlib import Path
from PIL import Image
import io

try:
    from langgraph.graph import StateGraph, END
    import vertexai
    from vertexai.generative_models import GenerativeModel, Part
    print("✅ 모든 라이브러리가 성공적으로 임포트되었습니다.")
except ImportError as e:
    print(f"❌ 필수 라이브러리가 누락되었습니다: {e}")
    print("pip install langgraph google-cloud-aiplatform vertexai 명령어로 설치해주세요.")

✅ 모든 라이브러리가 성공적으로 임포트되었습니다.


In [9]:
# 프로젝트 설정 (환경에 맞게 수정 필요)
PROJECT_ID = "p-01-emt-480312"  # 실제 프로젝트 ID로 변경
LOCATION = "us-central1"

# 모델 설정
MODEL_NAME = "gemini-2.5-pro"

# 시스템 인스트럭션
SYSTEM_INSTRUCTION = """1. 페르소나 및 기본 원칙

당신은 고도로 훈련된 시각 데이터 분석 전문가입니다.

당신은 모든 분석에서 '관찰'과 '해석'을 철저히 분리하며, 질문자의 유도 심리에 저항하고 오직 시각적 데이터에만 근거하여 답변합니다.

질문자가 특정 결론을 암시하거나 유도하더라도(예: "이것은 A가 맞죠?"), 시각적 증거가 뒷받침되지 않는다면 단호하게 중립을 유지합니다.

2. 분석 프로세스 (반드시 이 순서를 따를 것) 모든 사진 분석 요청에 대해 다음 4단계 구조로 답변하십시오.

Step 1. 객관적 관찰 (Observations): 이미지에서 보이는 물리적 사실만을 나열합니다. (예: 색상, 형태, 질감, 크기, 마모 상태, 기하학적 배치 등). 주관적인 형용사나 결론적인 단어를 배제하고 '현상'만 서술합니다.

Step 2. 논리적 해석 (Interpretation): 관찰된 사실이 어떤 물리적/과학적 원리와 연결될 수 있는지 분석합니다. 표준 사례(Reference)와의 일치점과 차이점을 논합니다.

Step 3. 최종 판단 및 확신도 (Conclusion & Confidence): 분석을 종합하여 결론을 내립니다. 이때 결론에 대한 확신도를 0~100% 사이로 표기하고, 확신할 수 없는 이유(변수)를 함께 기술합니다.

Step 4. 대안적 가능성 (Alternative Hypotheses): 현재 내린 결론 외에 발생할 수 있는 다른 가능성을 최소 한 가지 이상 제시합니다.

3. 불확실성 처리 규칙 (Negative Constraints)

확실하지 않은 정보에 대해서는 절대 추측하지 않습니다.

사진의 해상도, 각도, 조도 등으로 인해 식별이 어려운 경우, 아는 척하지 말고 반드시 **"시각적 정보 부족으로 판단 불가"**라고 명시하십시오.

시각적 증거가 100% 확보되지 않은 상태에서 "확실하다", "분명하다"라는 단어 사용을 지양합니다.

4. 답변 스타일

간결하고 구조화된 개조식(Bullet points)을 선호합니다.

감정적인 표현이나 부연 설명을 배제하고, 전문 용어를 정확하게 사용하되 필요시 정의를 덧붙입니다."""

def initialize_model():
    """Vertex AI Gemini 모델 초기화"""
    try:
        # 인증 확인 (로컬 실행 시 ADC 필요)
        # vertexai.init(project=PROJECT_ID, location=LOCATION)
        model = GenerativeModel(MODEL_NAME, system_instruction=SYSTEM_INSTRUCTION)
        print(f"✅ Vertex AI 모델 '{MODEL_NAME}' 초기화 완료")
        return model
    except Exception as e:
        print(f"⚠️ 모델 초기화 실패: {e}")
        print("💡 Application Default Credentials (ADC) 설정이 필요합니다:")
        print("   gcloud auth application-default login")
        return None

# 전역 모델 인스턴스
model = initialize_model()

✅ Vertex AI 모델 'gemini-2.5-pro' 초기화 완료


c:\Users\user\OneDrive\woRk\Development\Project\P_05_Scope\venv\Lib\site-packages\vertexai\generative_models\_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


In [10]:
def create_image_part(image_path: str) -> Optional[Part]:
    """Vertex AI용 이미지 Part 객체 생성"""
    if not os.path.exists(image_path):
        print(f"❌ 이미지 파일을 찾을 수 없습니다: {image_path}")
        return None
        
    try:
        with open(image_path, "rb") as image_file:
            image_data = image_file.read()
        
        # 확장자에 따른 MIME 타입 추론
        ext = Path(image_path).suffix.lower()
        mime_type = "image/png" if ext == ".png" else "image/jpeg"
        
        return Part.from_data(data=image_data, mime_type=mime_type)
    except Exception as e:
        print(f"❌ 이미지 로드 오류: {e}")
        return None

def call_gemini_vision(model: GenerativeModel, prompt: str, image_part: Part, step_name: str = "") -> tuple[str, Optional[Dict]]:
    """Gemini Vision API 호출 및 에러 핸들링
    
    Returns:
        tuple: (response_text, thinking_info)
    """
    try:
        response = model.generate_content([prompt, image_part])
        
        # Thinking 과정 추출 및 출력
        thinking_info = None
        response_text = ""
        full_response_text = ""
        
        if hasattr(response, 'candidates') and response.candidates:
            candidate = response.candidates[0]
            
            # 모든 파트 확인 (thinking 과정이 별도 파트로 있을 수 있음)
            if hasattr(candidate, 'content') and hasattr(candidate.content, 'parts'):
                parts = candidate.content.parts
                print(f"\n🔍 [{step_name}] 응답 파트 개수: {len(parts)}")
                
                all_texts = []
                for i, part in enumerate(parts):
                    if hasattr(part, 'text'):
                        part_text = part.text
                        all_texts.append(part_text)
                        if i == 0:
                            # 첫 번째 파트는 일반 응답
                            response_text = part_text
                        else:
                            # 이후 파트는 thinking 과정일 수 있음
                            if part_text and part_text.strip():
                                thinking_info = thinking_info or {}
                                thinking_info[f"part_{i}"] = part_text
                                print(f"\n💭 [{step_name}] 모델의 생각 과정 (파트 {i}):")
                                print("-" * 60)
                                print(part_text)
                                print("-" * 60)
                
                # 모든 파트의 텍스트를 합쳐서 전체 응답 확인
                full_response_text = "\n\n".join(all_texts)
            
            # 응답 객체의 모든 속성 확인 (디버깅용)
            print(f"\n📊 [{step_name}] 응답 객체 속성:")
            candidate_attrs = [attr for attr in dir(candidate) if not attr.startswith('_')]
            print(f"  - Candidate 속성: {', '.join(candidate_attrs[:10])}...")
            
            # Grounding metadata 확인
            if hasattr(candidate, 'grounding_metadata'):
                grounding = candidate.grounding_metadata
                if grounding:
                    thinking_info = thinking_info or {}
                    thinking_info["grounding"] = str(grounding)
                    print(f"\n📚 [{step_name}] Grounding 정보: {grounding}")
            
            # Finish reason 확인 (디버깅용)
            if hasattr(candidate, 'finish_reason'):
                finish_reason = candidate.finish_reason
                if finish_reason:
                    print(f"📋 [{step_name}] Finish reason: {finish_reason}")
        
        # response.text가 있으면 사용 (fallback)
        if not response_text and hasattr(response, 'text'):
            response_text = response.text
        
        # 전체 응답 텍스트 출력 (thinking 과정이 포함되어 있을 수 있음)
        if full_response_text and len(full_response_text) > len(response_text):
            print(f"\n💭 [{step_name}] 전체 응답 텍스트 (thinking 과정 포함 가능):")
            print("-" * 60)
            print(full_response_text[:2000])  # 처음 2000자만 출력
            if len(full_response_text) > 2000:
                print(f"... (총 {len(full_response_text)}자, 나머지 생략)")
            print("-" * 60)
            thinking_info = thinking_info or {}
            thinking_info["full_response"] = full_response_text
        
        # 응답 텍스트 항상 출력 (JSON 파싱 전에 전체 응답 확인)
        if response_text:
            print(f"\n💭 [{step_name}] 모델 응답 텍스트:")
            print("-" * 60)
            # JSON 시작 전까지의 텍스트 확인 (thinking 과정일 수 있음)
            json_start = response_text.find('{')
            if json_start > 0:
                thinking_part = response_text[:json_start].strip()
                if thinking_part:
                    print("📝 [Thinking 과정]:")
                    print(thinking_part)
                    print("\n📄 [JSON 응답]:")
                    print(response_text[json_start:json_start+500])
                    if len(response_text[json_start:]) > 500:
                        print(f"... (총 {len(response_text[json_start:])}자)")
                else:
                    print(response_text[:1000])
                    if len(response_text) > 1000:
                        print(f"... (총 {len(response_text)}자)")
            else:
                print(response_text[:1000])
                if len(response_text) > 1000:
                    print(f"... (총 {len(response_text)}자)")
            print("-" * 60)
        
        return response_text, thinking_info
    except Exception as e:
        print(f"❌ [{step_name}] API 호출 오류: {e}")
        import traceback
        traceback.print_exc()
        return f"Error: {str(e)}", None

def parse_json_response(response_text: str) -> Dict[str, Any]:
    """응답 텍스트에서 JSON 추출 및 파싱"""
    try:
        # JSON 부분만 추출 (마크다운 코드 블록 제거)
        json_start = response_text.find('{')
        json_end = response_text.rfind('}') + 1
        
        if json_start != -1 and json_end > json_start:
            json_text = response_text[json_start:json_end]
            return json.loads(json_text)
        else:
            print(f"⚠️ 유효한 JSON을 찾을 수 없습니다. 원본 응답:\n{response_text[:100]}...")
            return {"error": "JSON 파싱 실패", "raw_response": response_text}
    except json.JSONDecodeError as e:
        print(f"⚠️ JSON 디코딩 오류: {e}")
        return {"error": f"JSON 파싱 오류: {e}", "raw_response": response_text}

# Langgraph State 정의
class AgentState(TypedDict):
    """에이전트 상태 스키마"""
    image_path: str
    image_part: Optional[Part]
    step1_result: Optional[Dict]  # 소선 끝단 형상 분석 결과
    step2_result: Optional[Dict]  # 용융망울 크기 및 분포 분석 결과
    step3_result: Optional[Dict]  # 기계적 피로 흔적 분석 결과
    confidence_score: int  # 최종 신뢰도 점수 (0-100)
    analysis_summary: str  # 분석 요약
    evidence: List[Dict]   # 각 단계별 증거 수집

print("✅ 유틸리티 및 State 정의 완료")

✅ 유틸리티 및 State 정의 완료


In [11]:
# --- PROMPTS ---

STEP1_PROMPT = """당신은 금속 파단면 분석 및 전기 배선 손상 전문가입니다. 다음 이미지에서 전선 끝단의 소선 형상을 분석하여 반단선(Semi-disconnection) 여부를 판별하세요.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[단계별 분석 프로세스 (Chain of Thought)]
1단계: 시각적 요소 정밀 관찰
- 전선 끝단의 개별 소선(Strand) 형상을 확대하여 관찰하세요.
- 이미지의 해상도가 낮거나 각도가 애매하여 식별이 불가능한 경우, 절대 추측하지 말고 "unknown"으로 처리하세요.

2단계: 특징 식별 (Identification)
- 형상: 소선 끝이 바늘처럼 뾰족하게 가늘어지는지(Tapered) 확인하세요.
- 미세 용융: 뾰족해진 끝부분에 아주 작은 구슬(Micro-bead)이 맺혀 있는지 정밀하게 확인하세요. (이는 단순 인장 파괴와 반단선을 구별하는 핵심 증거입니다.)
- 열변색: 네킹이 일어난 부위 주변에 무지개빛이나 검은색의 국부적인 열변색 흔적이 있는지 확인하세요.
- 분리 여부: 소선들이 한 덩어리로 융착되지 않고 올올이 개별적으로 분리(Strand Separation)되어 있는지 확인하세요.

3단계: 논리적 추론
- 소선들이 개별적으로 분리되어 있고, 끝이 뾰족하면서(Necking) 끝단에 미세한 용융 흔적이 관찰된다면 반단선일 확률이 매우 높습니다.
- 용융 흔적 없이 뾰족하기만 하다면 기계적 인장 파괴일 가능성을, 거대한 망울이 있다면 단락일 가능성을 배제하지 마십시오.

[출력 형식]
다음 JSON 형식으로 응답하세요 (Unknown 값은 null이 아닌 문자열 "unknown"으로 표기하세요):
{
    "individual_strands_detected": true/false,
    "tapered_tips_detected": true/false,
    "micro_bead_at_tip": true/false,  // 끝단에 미세 용융망울 존재 여부
    "thermal_discoloration": true/false, // 열변색 존재 여부
    "necking_phenomenon": true/false,
    "tip_morphology": "tapered" | "blunt" | "fused" | "mixed" | "unknown",
    "strand_separation": true/false,
    "necking_description": "네킹, 미세 용융, 열변색에 대한 상세 관찰 내용",
    "confidence": 0-100,
    "reasoning": "판단 근거 요약"
}"""

STEP2_PROMPT = """당신은 금속 재료 공학 및 화재 감식 전문가입니다. 제공된 현미경 이미지를 분석하여 전선 용융흔(망울)의 형태학적 특징을 파악하는 것이 목표입니다.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[분석 목표]
이미지에서 '용융망울(Beads)'의 **크기**와 **분포**를 정밀하게 분석하여 반단선(Semi-disconnection) 여부를 판단할 수 있는 근거를 마련하십시오.

[단계별 분석 프로세스 (Chain of Thought)]
1단계: 시각적 요소 추출
2단계: 특징 서술
- 개별 소선(Strand) 끝마다 좁쌀 형태의 미세 망울(Micro-beads)이 있는지 확인하십시오.
- 망울들이 서로 뭉쳐있는지(Clustered) 개별적으로 산재해 있는지(Individual/Scattered) 확인하십시오.

3단계: 논리적 추론
- 반단선 아크는 에너지가 국부적이고 상대적으로 작아, 전선 전체를 녹이는 거대 망울보다는 소선 끝에 맺힌 미세 망울을 형성하는 경향이 있습니다.
- 관찰된 크기와 분포가 이 특징과 일치하는지 평가하십시오.

[출력 형식]
반드시 아래의 JSON 스키마를 준수하여 응답하십시오. Markdown 코드 블록(```json)을 포함하지 말고 순수 JSON 텍스트만 출력하는 것을 권장합니다.
{
    "micro_beads_detected": true, 
    "bead_size": "micro", 
    "bead_distribution": "individual_strands", 
    "bead_count": "many",
    "bead_description": "상세 관찰 내용...",
    "large_bead_present": false,
    "confidence": 95,
    "reasoning": "소선 끝마다 좁쌀 형태의 작은 망울들이 다수 관찰되며 거대 망울이 부재하므로..."
}"""

STEP3_PROMPT = """당신은 금속 파단면 분석 및 전기 배선 손상 전문가입니다. 제공된 현미경 이미지를 분석하여 전선의 '기계적 피로(Mechanical Fatigue)' 흔적을 식별하는 것이 목표입니다.

[중요] 먼저 분석 과정을 단계별로 자세히 설명한 후, 마지막에 JSON 형식으로 응답하세요. 각 단계에서 무엇을 관찰하고 어떻게 판단하는지 명확히 서술하세요.

[분석 목표]
이미지에서 전선 피복의 손상 상태와 굽힘/비틀림 흔적을 정밀하게 분석하여, 반단선(Semi-disconnection) 발생 가능성을 기계적 관점에서 판단하십시오.

[단계별 분석 프로세스 (Chain of Thought)]
1단계: 시각적 요소 추출
- 전선 피복(Insulation)에 균열(Cracking), 마모(Wear), 또는 소성 변형(Deformation)이 있는지 객관적으로 관찰하십시오.
2단계: 특징 서술
- 손상 부위가 스트레인 릴리프(Strain relief, 플러그 목 부분)나 자주 꺾이는 굴곡점(Bend point)인지 확인하십시오.
- 해당 위치에서 소선들이 끊어졌는지 확인하여 기계적 스트레스와의 연관성을 파악하십시오.
3단계: 논리적 추론
- 반단선은 주로 코드가 자주 꺾이거나 비틀리는 부분에서 기계적 피로 누적으로 인해 발생합니다.
- 피로 흔적의 위치(피복 손상 부위)와 소선 파단 위치가 일치한다면 반단선 가능성이 매우 높습니다.

[출력 형식]
반드시 아래의 JSON 스키마를 준수하여 응답하십시오. Markdown 코드 블록(```json)을 포함하지 말고 순수 JSON 텍스트만 출력하는 것을 권장합니다.
{
    "mechanical_fatigue_detected": true,
    "fatigue_location": "strain_relief",
    "insulation_damage": true,
    "insulation_damage_type": "cracking",
    "bending_evidence": true,
    "location_match": true,
    "fatigue_description": "플러그 목 부분의 피복에 깊은 균열이 관찰되며...",
    "confidence": 92,
    "reasoning": "스트레인 릴리프 부위의 피복 균열과 소선 파단 위치가 일치하여 반복적 굽힘에 의한 피로 파괴로 판단됨."
}"""

# --- NODES ---

def step1_tip_morphology(state: AgentState) -> AgentState:
    """Step 1: 소선 끝단의 형상 분석"""
    print("\n🔍 [Step 1] 소선 끝단의 형상 분석 시작...")
    
    if state.get("image_part") is None:
        state["image_part"] = create_image_part(state["image_path"])
        if state["image_part"] is None:
            return {**state, "step1_result": {"error": "이미지 로드 실패"}}

    response_text, thinking_info = call_gemini_vision(model, STEP1_PROMPT, state["image_part"], "Step 1")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    tapered_tips = result.get("tapered_tips_detected", False)
    print(f"✅ [Step 1] 완료: 뾰족한 끝단 {'탐지됨' if tapered_tips else '미탐지'}")
    
    return {
        **state,
        "step1_result": result
    }

def step2_bead_distribution(state: AgentState) -> AgentState:
    """Step 2: 용융망울의 크기와 분포 분석"""
    print("\n🔍 [Step 2] 용융망울의 크기와 분포 분석 시작...")
    
    if state.get("image_part") is None:
        return {**state, "step2_result": {"error": "이미지 없음"}}

    response_text, thinking_info = call_gemini_vision(model, STEP2_PROMPT, state["image_part"], "Step 2")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    micro_beads = result.get("micro_beads_detected", False)
    print(f"✅ [Step 2] 완료: 미세 망울 {'탐지됨' if micro_beads else '미탐지'}")

    return {
        **state,
        "step2_result": result
    }

def step3_mechanical_fatigue(state: AgentState) -> AgentState:
    """Step 3: 기계적 피로 흔적 분석"""
    print("\n🔍 [Step 3] 기계적 피로 흔적 분석 시작...")
    
    if state.get("image_part") is None:
        return {**state, "step3_result": {"error": "이미지 없음"}}

    response_text, thinking_info = call_gemini_vision(model, STEP3_PROMPT, state["image_part"], "Step 3")
    result = parse_json_response(response_text)
    
    # Thinking 정보를 결과에 추가
    if thinking_info:
        result["thinking_process"] = thinking_info
    
    fatigue = result.get("mechanical_fatigue_detected", False)
    print(f"✅ [Step 3] 완료: 기계적 피로 {'탐지됨' if fatigue else '미탐지'}")

    return {
        **state,
        "step3_result": result
    }

def final_judgment(state: AgentState) -> AgentState:
    """최종 판정: 3단계 결과 종합 및 신뢰도 점수 계산"""
    print("\n⚖️ [Final Judgment] 최종 판정 시작...")
    
    step1 = state.get("step1_result", {}) or {}
    step2 = state.get("step2_result", {}) or {}
    step3 = state.get("step3_result", {}) or {}
    
    # 각 단계별 점수 추출
    step1_score = step1.get("confidence", 0) if not step1.get("error") else 0
    step2_score = step2.get("confidence", 0) if not step2.get("error") else 0
    step3_score = step3.get("confidence", 0) if not step3.get("error") else 0
    
    # 핵심 지표 확인
    individual_strands = step1.get("individual_strands_detected", False)
    tapered_tips = step1.get("tapered_tips_detected", False)
    necking_phenomenon = step1.get("necking_phenomenon", False)
    micro_beads = step2.get("micro_beads_detected", False)
    no_large_bead = not step2.get("large_bead_present", True)
    mechanical_fatigue = step3.get("mechanical_fatigue_detected", False)
    location_match = step3.get("location_match", False)
    
    # 신뢰도 점수 계산 (가중치 적용)
    base_score = 0
    
    # 핵심 지표 가중치
    if individual_strands: base_score += 20
    if tapered_tips: base_score += 30  # 뾰족한 끝단이 가장 중요한 증거
    if necking_phenomenon: base_score += 25
    if micro_beads: base_score += 20
    if no_large_bead: base_score += 10
    if mechanical_fatigue: base_score += 15
    if location_match: base_score += 10
    
    # 각 단계별 신뢰도 점수의 평균 반영 (10%)
    avg_confidence = (step1_score + step2_score + step3_score) / 3
    base_score += avg_confidence * 0.1
    
    # 핵심 조합에 따른 보정
    if tapered_tips and micro_beads and mechanical_fatigue:
        base_score = max(base_score, 90)
    
    if necking_phenomenon and micro_beads:
        base_score = max(base_score, 85)
    
    final_score = min(100, max(0, int(base_score)))
    
    # 증거 수집
    evidence = []
    if individual_strands:
        evidence.append({"step": 1, "evidence": "개별 소선 확인", "details": step1.get("necking_description", "")})
    if tapered_tips:
        evidence.append({"step": 1, "evidence": "뾰족한 끝단 확인", "details": step1.get("necking_description", "")})
    if necking_phenomenon:
        evidence.append({"step": 1, "evidence": "네킹 현상 확인", "details": step1.get("necking_description", "")})
    if micro_beads:
        evidence.append({"step": 2, "evidence": "미세 망울 확인", "details": step2.get("bead_description", "")})
    if mechanical_fatigue:
        evidence.append({"step": 3, "evidence": "기계적 피로 확인", "details": step3.get("fatigue_description", "")})
    if location_match:
        evidence.append({"step": 3, "evidence": "위치 일치 확인", "details": step3.get("fatigue_location", "")})
    
    # 분석 요약 생성
    summary_parts = [f"반단선 판정 신뢰도: {final_score}%"]
    summary_parts.append(f"✓ 뾰족한 끝단 확인: {step1.get('tip_morphology', 'unknown')}" if tapered_tips else "✗ 뾰족한 끝단 미확인")
    summary_parts.append("✓ 네킹 현상 확인" if necking_phenomenon else "✗ 네킹 현상 미확인")
    summary_parts.append(f"✓ 미세 망울 확인: {step2.get('bead_size', 'unknown')}" if micro_beads else "✗ 미세 망울 미확인")
    summary_parts.append(f"✓ 기계적 피로 확인: {step3.get('fatigue_location', 'unknown')}" if mechanical_fatigue else "✗ 기계적 피로 미확인")
    summary_parts.append("✓ 위치 일치 확인" if location_match else "✗ 위치 일치 미확인")
    
    analysis_summary = "\n".join(summary_parts)
    print(f"✅ [Final Judgment] 완료: 신뢰도 {final_score}%")
    
    return {
        **state,
        "confidence_score": final_score,
        "analysis_summary": analysis_summary,
        "evidence": evidence
           }

In [12]:
def create_agent_graph():
    """반단선 판별 에이전트 그래프 생성"""
    workflow = StateGraph(AgentState)
    
    # 노드 추가
    workflow.add_node("step1_tip", step1_tip_morphology)
    workflow.add_node("step2_bead", step2_bead_distribution)
    workflow.add_node("step3_fatigue", step3_mechanical_fatigue)
    workflow.add_node("final_judgment", final_judgment)
    
    # 엣지 연결 (순차 실행)
    workflow.set_entry_point("step1_tip")
    workflow.add_edge("step1_tip", "step2_bead")
    workflow.add_edge("step2_bead", "step3_fatigue")
    workflow.add_edge("step3_fatigue", "final_judgment")
    workflow.add_edge("final_judgment", END)
    
    return workflow.compile()

# 전역 그래프 객체
try:
    agent_app = create_agent_graph()
    print("✅ Langgraph StateGraph 구성 완료")
except Exception as e:
    print(f"⚠️ 그래프 구성 실패 (Langgraph 미설치 등): {e}")
    agent_app = None

def analyze_semi_disconnection(image_path: str) -> dict:
    """전체 반단선 분석 실행 함수"""
    if agent_app is None:
        return {"error": "Agent 그래프가 초기화되지 않았습니다."}

    initial_state: AgentState = {
        "image_path": image_path,
        "image_part": None,
        "step1_result": None,
        "step2_result": None,
        "step3_result": None,
        "confidence_score": 0,
        "analysis_summary": "",
        "evidence": []
    }
    
    print(f"\n{'='*60}\n🔍 반단선 분석 시작: {image_path}\n{'='*60}")
    
    try:
        final_state = agent_app.invoke(initial_state)
        return {
            "confidence_score": final_state["confidence_score"],
            "analysis_summary": final_state["analysis_summary"],
            "step1_result": final_state["step1_result"],
            "step2_result": final_state["step2_result"],
            "step3_result": final_state["step3_result"],
            "evidence": final_state["evidence"]
        }
    except Exception as e:
        print(f"❌ 분석 중 오류 발생: {e}")
        return {"error": str(e)}

✅ Langgraph StateGraph 구성 완료


In [13]:
def test_single_step(step_name: Literal["step1", "step2", "step3"], image_path: str, prev_state: Optional[AgentState] = None):
    """
    특정 단계만 독립적으로 테스트하기 위한 함수
    """
    print(f"\n🧪 [Test] {step_name} 독립 실행 테스트 중...")
    
    if prev_state:
        state = prev_state.copy()
    else:
        state: AgentState = {
            "image_path": image_path,
            "image_part": None, # 노드 내부에서 생성됨
            "step1_result": None,
            "step2_result": None,
            "step3_result": None,
            "confidence_score": 0,
            "analysis_summary": "",
            "evidence": []
        }
    
    try:
        if step_name == "step1":
            result_state = step1_tip_morphology(state)
            print("결과:", json.dumps(result_state["step1_result"], indent=2, ensure_ascii=False))
        elif step_name == "step2":
            result_state = step2_bead_distribution(state)
            print("결과:", json.dumps(result_state["step2_result"], indent=2, ensure_ascii=False))
        elif step_name == "step3":
            result_state = step3_mechanical_fatigue(state)
            print("결과:", json.dumps(result_state["step3_result"], indent=2, ensure_ascii=False))
        return result_state
    except Exception as e:
        print(f"❌ 테스트 실패: {e}")
        return None

In [14]:
if __name__ == "__main__":
    # 테스트할 이미지 경로 설정 (노트북 기준 상대 경로)
    TEST_IMAGE_PATH = "../data/Cu2O_Breeding.jpg"
    
    # 이미지 파일 존재 여부 확인
    if not os.path.exists(TEST_IMAGE_PATH):
        print(f"⚠️ 경고: 테스트 이미지 '{TEST_IMAGE_PATH}'가 없습니다. 경로를 확인하세요.")
    else:
        # 전체 분석 실행
        result = analyze_semi_disconnection(TEST_IMAGE_PATH)
        
        # 결과 출력
        print("\n" + "="*60)
        print("📊 분석 결과")
        print("="*60)
        print(result.get("analysis_summary", ""))
        print(f"\n신뢰도 점수: {result.get('confidence_score', 0)}%")
        print("\n증거:")
        for ev in result.get("evidence", []):
            print(f"  - Step {ev.get('step')}: {ev.get('evidence')}")


🔍 반단선 분석 시작: ../data/Cu2O_Breeding.jpg

🔍 [Step 1] 소선 끝단의 형상 분석 시작...

🔍 [Step 1] 응답 파트 개수: 1

📊 [Step 1] 응답 객체 속성:
  - Candidate 속성: avg_logprobs, citation_metadata, content, finish_message, finish_reason, from_dict, function_calls, grounding_metadata, index, logprobs_result...
📋 [Step 1] Finish reason: 1

💭 [Step 1] 모델 응답 텍스트:
------------------------------------------------------------
📝 [Thinking 과정]:
알겠습니다. 시각 데이터 분석 전문가로서, 제공된 이미지의 전선 손상 형태를 단계별로 분석하고 최종적으로 JSON 형식으로 결과를 제출하겠습니다.

### 분석 프로세스

#### Step 1. 객관적 관찰 (Observations)

*   이미지에는 녹색 배경 위에 두 개의 구리 전선이 놓여 있습니다.
*   **굵은 연선 (Twisted Wire):**
    *   여러 가닥의 구리 소선(strand)이 꼬여있는 구조입니다.
    *   **좌측 끝단:** 소선들이 하나로 녹아 붙어 불규칙한 형태의 덩어리(glob)를 형성하고 있습니다. 표면에는 녹색 및 검은색 물질이 부착되어 있습니다. 개별 소선의 형태는 식별되지 않습니다.
    *   **중간 부분:** 와이어 전체에 걸쳐 검은색으로 변색된 부분이 다수 관찰됩니다. 일부 영역에서는 구리 본래의 색이 보입니다.
    *   **우측 끝단:** 베이지색 피복이 불에 타서 파괴되었고, 내부의 구리 소선들이 노출되어 있습니다. 소선들은 일부 끊어지고 흩어져 있으나, 심한 탄화 및 낮은 해상도로 인해 개별 소선의 끝단 형상(tip morphology)을 정밀하게 식별하기는 어렵습니다